# Pipeline EHBG-FACS · 03 · NCO supervisado (Attention Model)

**Paradigma 3 — Attention Model entrenado por imitación de un maestro.**

Un **Transformer codificador-decodificador** (Kool/Kwon) entrenado por *teacher forcing* para imitar rutas etiquetadas por un **maestro** (caro): `exact-bc` (óptimo, ignora ventanas) en `nco-sl`, o `aco` (factible) en `nco-sl-feas`. La (in)factibilidad la define el maestro. Inferencia en milisegundos; entrenamiento amortizado en GPU.

In [ ]:
# === Configuración del entorno (ejecuta esta celda primero) =================
# Requiere: (a) el paquete `svrplab` (carpeta experiments/colab del repo de tesis)
#           (b) el repo oficial de SVRPBench (se clona solo en bootstrap.init()).
REPO_URL  = "https://github.com/AbrahaHub/TESIS-ANT"   # <-- EDITA si tu repo difiere
USE_DRIVE = True   # persistir banco/resultados/modelos en Google Drive (recomendado)

import os, sys, subprocess

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive no disponible (¿ejecutas local?):", e)

def _find_svrplab():
    cands = ["/content/drive/MyDrive/TESIS-ANT/experiments/colab",
             "/content/TESIS-ANT/experiments/colab",
             os.path.join(os.getcwd(), "experiments", "colab"),
             os.getcwd()]
    for c in cands:
        if os.path.isdir(os.path.join(c, "svrplab")):
            return c
    return None

_path = _find_svrplab()
if _path is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TESIS-ANT"], check=False)
    _path = "/content/TESIS-ANT/experiments/colab"
sys.path.insert(0, _path)
print("svrplab en:", _path)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "pandas",
                "matplotlib", "scikit-learn", "pillow", "tqdm"], check=False)
# torch ya viene en Colab. gurobipy solo se instala en el notebook del paradigma 1.

from svrplab import bootstrap, protocol, data, runner, metrics, viz
env   = bootstrap.init()        # GPU + repo oficial SVRPBench + rutas (Drive si está montado)
proto = protocol.DEFAULT
print("device:", env.device, "| raíz de artefactos:", env.paths.root)

In [ ]:
# === Configuración del experimento (IDÉNTICA en los 5 notebooks) ============
# Para garantizar el "piso parejo", TODOS los notebooks deben usar los MISMOS
# SIZES y N_INSTANCES: así resuelven exactamente el mismo banco de instancias.
SIZES       = [10, 20, 50]           # clientes. Extiende a [50,100,200,300] (ver notas).
N_INSTANCES = proto.instances_per_size   # 30 (rigor estadístico). Corrida rápida: pon 5.

bank = data.load_bank(env.paths.instances, SIZES, N_INSTANCES,
                      base_seed=proto.base_seed, capacity_mode=proto.capacity_mode, verbose=True)
print({s: len(v) for s, v in bank.items()}, "instancias por tamaño")

## Entrenar e inferir (GPU)
Las etiquetas se generan resolviendo instancias de entrenamiento con el maestro (requiere Gurobi si el maestro es `exact-bc`). El modelo se cachea en Drive. Ajusta `epochs`/`n_per_size` según el tiempo disponible.

In [ ]:
# Maestro exact-bc requiere Gurobi:
import subprocess, sys; subprocess.run([sys.executable,"-m","pip","install","-q","gurobipy"], check=False)
from svrplab.solvers.nco_sl import NCOSupervised, NCOSupervisedFeasible
import pandas as pd
common = dict(train_sizes=(10,20), n_per_size=256, epochs=80, embed_dim=128,
              device=env.device, models_dir=env.paths.models, verbose=True)
sl      = NCOSupervised(teacher="exact-bc", **common)
sl_feas = NCOSupervisedFeasible(**common)   # maestro = aco (factible)
df_sl   = runner.run_solver(sl,      "nco-sl",      bank, env, proto, verbose=True)
df_feas = runner.run_solver(sl_feas, "nco-sl-feas", bank, env, proto, verbose=True)
df = pd.concat([df_sl, df_feas], ignore_index=True); df

## Curva de entrenamiento y figuras

In [ ]:
import matplotlib.pyplot as plt
if getattr(sl, "history", None):
    viz.plot_training_curve(sl.history, ylabel="CE", title="nco-sl: pérdida de imitación"); plt.show()
display(metrics.aggregate_by_size(df))
viz.plot_comparison(df); plt.show()

**Interpretación.** `nco-sl` (imita al óptimo) hereda baja factibilidad; `nco-sl-feas` (imita a aco) hereda factibilidad alta — la imitación voraz es **con pérdida**. Confirma que el límite proviene del **maestro**, no del paradigma NCO.